# 📋 Tugas UAS Mata Kuliah Kecerdasan Buatan (A)

| | |
| :--- | :--- |
| **Nama** | Muhammad Fathir Alfarqi |
| **NPM** | 4524210064 |
| **Dataset** | Healthcare Dataset - Stroke Prediction (Kaggle) |
| **Jumlah Data** | 5.110 data pasien, 12 fitur |
| **Framework** | CRISP-DM (Cross-Industry Standard Process for Data Mining) |

---

### Deskripsi Singkat
Proyek ini bertujuan untuk membangun model **klasifikasi** berbasis Machine Learning yang dapat memprediksi apakah seorang pasien memiliki **risiko terkena penyakit stroke** berdasarkan data rekam medis.

Dataset yang digunakan berasal dari Kaggle dengan nama *"Healthcare Dataset - Stroke Data"*, berisi **5.110 data pasien** dengan **12 fitur** seperti usia, jenis kelamin, riwayat hipertensi, penyakit jantung, kadar glukosa rata-rata, BMI, dan status merokok.

Alur pengerjaan mengikuti kerangka kerja **CRISP-DM** yang terdiri dari 6 tahap:
1. **Business Understanding** — Memahami tujuan dan permasalahan bisnis/medis
2. **Data Understanding** — Eksplorasi dan visualisasi data (EDA)
3. **Data Preparation** — Pembersihan, imputasi, dan encoding data
4. **Modeling** — Pembagian data dan pelatihan algoritma klasifikasi
5. **Evaluation** — Evaluasi performa model menggunakan Confusion Matrix
6. **Deployment** — Penerapan model ke dalam aplikasi web interaktif (Streamlit)

---
## Tahap 1: Business Understanding

**Latar Belakang:**
Stroke adalah penyebab kematian kedua terbesar di dunia menurut World Health Organization (WHO), bertanggung jawab atas sekitar 11% dari total kematian global. Deteksi dini terhadap risiko stroke sangat penting agar pasien dapat segera mendapatkan penanganan yang tepat.

**Tujuan Proyek:**
Membangun model Machine Learning yang dapat memprediksi apakah seorang pasien berisiko terkena stroke berdasarkan data medis seperti usia, hipertensi, penyakit jantung, kadar glukosa, BMI, dan status merokok.

**Kriteria Keberhasilan:**
Model harus memiliki **Recall yang tinggi** untuk kelas Stroke, karena dalam konteks medis lebih berbahaya jika model gagal mendeteksi pasien yang sebenarnya berisiko stroke (False Negative) dibandingkan salah menandai pasien sehat sebagai berisiko (False Positive).

---
## Tahap 2: Data Understanding (Eksplorasi Data / EDA)
Pada tahap ini, kita memuat dataset, melihat struktur data, dan membuat visualisasi untuk memahami pola dan karakteristik data.

In [ ]:
# Import seluruh library yang dibutuhkan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import io, base64, zlib

# Library Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score,
                             recall_score, precision_score, f1_score)

import warnings
warnings.filterwarnings('ignore')

print("Semua library berhasil diimpor!")

In [ ]:
# Dataset sudah di-embed langsung di dalam notebook ini
# sehingga tidak perlu upload file CSV secara terpisah.
EMBEDDED_DATA = "..."

# Decompress dan baca sebagai DataFrame
csv_bytes = zlib.decompress(base64.b64decode(EMBEDDED_DATA))
df = pd.read_csv(io.BytesIO(csv_bytes))

print(f"Dataset berhasil dimuat!")
print(f"Jumlah data: {df.shape[0]} baris, {df.shape[1]} kolom")
print()
df.head()

In [ ]:
# Melihat informasi struktur dan tipe data
print("Informasi Struktur Dataset:")
df.info()

In [ ]:
# Statistik deskriptif untuk kolom numerik
print("Statistik Deskriptif Data Numerik:")
df.describe()

**Penjelasan:**
- Dataset terdiri dari **5.110 data pasien** dengan **12 kolom** (11 fitur + 1 target `stroke`).
- Kolom `bmi` memiliki **201 data kosong (NaN)** yang perlu ditangani di tahap Data Preparation.
- Kolom `id` tidak relevan untuk prediksi dan akan dihapus.
- Terdapat 5 kolom bertipe teks (kategorikal) yang perlu diubah menjadi angka (encoding).

### 2.1 Visualisasi Distribusi Kelas Target (Stroke vs Sehat)

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='stroke', data=df, palette='Set2')
plt.title('Distribusi Kelas Target (Stroke)', fontsize=14, fontweight='bold')
plt.xlabel('Status Stroke (0 = Sehat, 1 = Stroke)')
plt.ylabel('Jumlah Pasien')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
               ha='center', va='bottom', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

stroke_count = df['stroke'].value_counts()
print(f"Jumlah pasien Sehat  (0): {stroke_count[0]} ({stroke_count[0]/len(df)*100:.1f}%)")
print(f"Jumlah pasien Stroke (1): {stroke_count[1]} ({stroke_count[1]/len(df)*100:.1f}%)")

**Kesimpulan:** Dataset sangat **tidak seimbang (imbalanced)** — hanya ~4.9% data yang merupakan kasus stroke. Hal ini perlu diatasi saat modeling agar model tidak bias hanya memprediksi kelas mayoritas (sehat).

### 2.2 Distribusi Usia terhadap Stroke

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='age', hue='stroke', kde=True, multiple='stack', palette='coolwarm')
plt.title('Distribusi Usia Pasien terhadap Kejadian Stroke', fontsize=14, fontweight='bold')
plt.xlabel('Usia (Tahun)')
plt.ylabel('Jumlah')
plt.tight_layout()
plt.show()

**Kesimpulan:** Kasus stroke secara signifikan lebih banyak terjadi pada pasien berusia **di atas 50 tahun**. Usia merupakan salah satu faktor risiko utama penyakit stroke.

### 2.3 Distribusi Body Mass Index (BMI)

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['bmi'].dropna(), kde=True, color='purple')
plt.title('Distribusi Body Mass Index (BMI)', fontsize=14, fontweight='bold')
plt.xlabel('BMI')
plt.ylabel('Frekuensi')
plt.tight_layout()
plt.show()

**Kesimpulan:** Distribusi BMI mendekati distribusi normal dengan sedikit kemiringan ke kanan (*right-skewed*). Mayoritas pasien memiliki BMI antara 23–33, yang menunjukkan sebagian besar pasien memiliki berat badan berlebih.

### 2.4 Distribusi Kadar Glukosa terhadap Stroke

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='avg_glucose_level', hue='stroke', kde=True, multiple='stack', palette='viridis')
plt.title('Distribusi Kadar Glukosa terhadap Kejadian Stroke', fontsize=14, fontweight='bold')
plt.xlabel('Rata-rata Kadar Glukosa (mg/dL)')
plt.ylabel('Jumlah')
plt.tight_layout()
plt.show()

**Kesimpulan:** Pasien dengan kadar glukosa tinggi (di atas 150 mg/dL) memiliki proporsi kasus stroke yang lebih besar. Kadar glukosa tinggi berkaitan dengan diabetes yang merupakan faktor risiko stroke.

### 2.5 Heatmap Korelasi Antar Fitur Numerik

In [ ]:
plt.figure(figsize=(8, 6))
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation = df[numeric_cols].corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Heatmap Korelasi Antar Fitur Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Kesimpulan:** Dari heatmap korelasi, fitur `age` (usia) memiliki korelasi tertinggi terhadap `stroke`. Fitur lain seperti `hypertension`, `heart_disease`, dan `avg_glucose_level` juga menunjukkan korelasi positif meskipun kecil.

---
## Tahap 3: Data Preparation
Pada tahap ini, data mentah dibersihkan dan disiapkan agar algoritma Machine Learning dapat memprosesnya. Langkah-langkah yang dilakukan:
1. Mengisi nilai kosong (missing values) pada kolom BMI dengan **median**
2. Menghapus data tidak representatif (gender 'Other', hanya 1 data)
3. Menghapus kolom yang tidak relevan (`id`)
4. Mengubah data kategorikal menjadi numerik (**One-Hot Encoding**)

In [ ]:
# Cek jumlah data kosong (missing values) per kolom
print("Jumlah data kosong per kolom SEBELUM pembersihan:")
print(df.isnull().sum())
print(f"\nTotal data kosong: {df.isnull().sum().sum()} dari {df.shape[0]} baris")

In [ ]:
# 1. Imputasi (mengisi) nilai kosong kolom BMI dengan MEDIAN
#    Median dipilih karena lebih tahan terhadap outlier dibanding mean
median_bmi = df['bmi'].median()
df['bmi'].fillna(median_bmi, inplace=True)
print(f"Nilai kosong pada kolom BMI diisi dengan median: {median_bmi}")

# 2. Hapus baris dengan gender 'Other' (hanya 1 baris, tidak representatif)
df = df[df['gender'] != 'Other']
print(f"Baris dengan gender 'Other' telah dihapus.")

# 3. Hapus kolom 'id' karena tidak memiliki daya prediksi
df = df.drop(['id'], axis=1)
print(f"Kolom 'id' telah dihapus.")

print(f"\nUkuran data setelah pembersihan: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"\nJumlah data kosong SETELAH pembersihan:")
print(df.isnull().sum())

In [ ]:
# 4. Encoding kolom Kategorikal (mengubah teks menjadi angka)
#    Menggunakan teknik One-Hot Encoding
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
print(f"Kolom kategorikal yang akan di-encode: {categorical_cols}")

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Konversi kolom boolean menjadi integer (True=1, False=0)
for col in df_encoded.columns:
    if df_encoded[col].dtype == 'bool':
        df_encoded[col] = df_encoded[col].astype(int)

print(f"\nJumlah kolom setelah encoding: {df_encoded.shape[1]}")
print(f"Kolom baru: {list(df_encoded.columns)}")
df_encoded.head()

**Kesimpulan Data Preparation:**
- 201 data kosong pada kolom `bmi` berhasil diisi menggunakan nilai **median (28.1)**.
- Kolom `id` dihapus karena hanya berisi nomor identitas pasien yang tidak relevan untuk prediksi.
- 5 kolom kategorikal berhasil diubah menjadi numerik menggunakan **One-Hot Encoding**, sehingga total kolom bertambah menjadi 16 kolom.

---
## Tahap 4: Modeling (Pemodelan)
Pada tahap ini, data dibagi menjadi **data latih (train)** dan **data uji (test)**, kemudian dilatih menggunakan dua algoritma klasifikasi:
1. **Decision Tree** — Algoritma berbasis pohon keputusan
2. **Random Forest** — Ensemble dari banyak pohon keputusan (lebih stabil)

In [ ]:
# Memisahkan Fitur (X) dan Target (y)
X = df_encoded.drop('stroke', axis=1)  # Semua kolom kecuali 'stroke'
y = df_encoded['stroke']               # Kolom target: stroke (0 atau 1)

print(f"Jumlah fitur (X): {X.shape[1]} kolom")
print(f"Jumlah data  (y): {y.shape[0]} baris")
print(f"\nDistribusi kelas target:")
print(f"  Sehat  (0): {(y==0).sum()} baris")
print(f"  Stroke (1): {(y==1).sum()} baris")

In [ ]:
# Membagi data: 80% untuk training, 20% untuk testing
# stratify=y memastikan proporsi stroke tetap sama di kedua set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Data Latih (Train): {X_train.shape[0]} baris ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Data Uji   (Test) : {X_test.shape[0]} baris ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\nProporsi Stroke di Train: {y_train.mean()*100:.1f}%")
print(f"Proporsi Stroke di Test : {y_test.mean()*100:.1f}%")

**Penjelasan Split Data:**
- Data dibagi dengan rasio **80:20** — 80% untuk melatih model, 20% untuk menguji performa model.
- Parameter `stratify=y` memastikan proporsi kelas Stroke tetap sama (~4.9%) di kedua set agar hasil evaluasi akurat.
- Parameter `random_state=42` digunakan agar hasil pembagian data **dapat direproduksi** (selalu sama setiap kali dijalankan).

In [ ]:
# ============================
# MODEL 1: DECISION TREE
# ============================
# class_weight='balanced' memberi bobot lebih pada kelas minoritas (Stroke)
# max_depth=5 membatasi kedalaman pohon agar tidak overfitting
dt_model = DecisionTreeClassifier(
    random_state=42,
    class_weight='balanced',
    max_depth=5
)
dt_model.fit(X_train, y_train)
print("Model 1 (Decision Tree) berhasil dilatih!")

# ============================
# MODEL 2: RANDOM FOREST
# ============================
# n_estimators=100 berarti menggunakan 100 pohon keputusan
rf_model = RandomForestClassifier(
    random_state=42,
    class_weight='balanced',
    n_estimators=100,
    max_depth=5
)
rf_model.fit(X_train, y_train)
print("Model 2 (Random Forest) berhasil dilatih!")

print("\nKedua model telah selesai dilatih dan siap dievaluasi.")

**Penjelasan Modeling:**
- Kedua model menggunakan parameter `class_weight='balanced'` yang secara otomatis memberikan **bobot lebih besar pada kelas minoritas** (Stroke). Ini penting karena dataset kita sangat tidak seimbang.
- `max_depth=5` digunakan untuk membatasi kedalaman pohon agar model tidak terlalu kompleks (*overfitting*).

---
## Tahap 5: Evaluation (Evaluasi Model)
Pada tahap ini, kita mengukur performa kedua model menggunakan data uji (test set) yang belum pernah dilihat oleh model. Metrik yang digunakan:
- **Accuracy** — Persentase prediksi benar secara keseluruhan
- **Precision** — Dari yang diprediksi positif (stroke), berapa yang benar-benar positif
- **Recall** — Dari yang sebenarnya positif (stroke), berapa yang berhasil terdeteksi
- **Confusion Matrix** — Tabel perbandingan prediksi vs data aktual

In [ ]:
# ============================
# EVALUASI MODEL 1: DECISION TREE
# ============================
y_pred_dt = dt_model.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)

print("=" * 55)
print("       EVALUASI MODEL 1: DECISION TREE")
print("=" * 55)
print(f"Akurasi: {acc_dt*100:.2f}%\n")
print(classification_report(y_test, y_pred_dt, target_names=['Sehat (0)', 'Stroke (1)']))

In [ ]:
# Confusion Matrix - Decision Tree
cm_dt = confusion_matrix(y_test, y_pred_dt)
disp_dt = ConfusionMatrixDisplay(confusion_matrix=cm_dt, display_labels=['Sehat', 'Stroke'])
fig, ax = plt.subplots(figsize=(6, 5))
disp_dt.plot(cmap='Blues', ax=ax)
plt.title('Confusion Matrix - Decision Tree', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nCara Membaca Confusion Matrix:")
print(f"  - True Negative  (TN): {cm_dt[0][0]} pasien sehat diprediksi sehat (BENAR)")
print(f"  - False Positive (FP): {cm_dt[0][1]} pasien sehat diprediksi stroke (SALAH)")
print(f"  - False Negative (FN): {cm_dt[1][0]} pasien stroke diprediksi sehat (SALAH - BERBAHAYA)")
print(f"  - True Positive  (TP): {cm_dt[1][1]} pasien stroke diprediksi stroke (BENAR)")

In [ ]:
# ============================
# EVALUASI MODEL 2: RANDOM FOREST
# ============================
y_pred_rf = rf_model.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

print("=" * 55)
print("       EVALUASI MODEL 2: RANDOM FOREST")
print("=" * 55)
print(f"Akurasi: {acc_rf*100:.2f}%\n")
print(classification_report(y_test, y_pred_rf, target_names=['Sehat (0)', 'Stroke (1)']))

In [ ]:
# Confusion Matrix - Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=['Sehat', 'Stroke'])
fig, ax = plt.subplots(figsize=(6, 5))
disp_rf.plot(cmap='Purples', ax=ax)
plt.title('Confusion Matrix - Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nCara Membaca Confusion Matrix:")
print(f"  - True Negative  (TN): {cm_rf[0][0]} pasien sehat diprediksi sehat (BENAR)")
print(f"  - False Positive (FP): {cm_rf[0][1]} pasien sehat diprediksi stroke (SALAH)")
print(f"  - False Negative (FN): {cm_rf[1][0]} pasien stroke diprediksi sehat (SALAH - BERBAHAYA)")
print(f"  - True Positive  (TP): {cm_rf[1][1]} pasien stroke diprediksi stroke (BENAR)")

### Perbandingan Kedua Model

In [ ]:
# Tabel perbandingan hasil evaluasi
comparison = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest'],
    'Accuracy': [f'{accuracy_score(y_test, y_pred_dt)*100:.2f}%', f'{accuracy_score(y_test, y_pred_rf)*100:.2f}%'],
    'Precision (Stroke)': [f'{precision_score(y_test, y_pred_dt)*100:.2f}%', f'{precision_score(y_test, y_pred_rf)*100:.2f}%'],
    'Recall (Stroke)': [f'{recall_score(y_test, y_pred_dt)*100:.2f}%', f'{recall_score(y_test, y_pred_rf)*100:.2f}%'],
    'F1-Score (Stroke)': [f'{f1_score(y_test, y_pred_dt)*100:.2f}%', f'{f1_score(y_test, y_pred_rf)*100:.2f}%']
})
print("Tabel Perbandingan Performa Kedua Model:")
print("=" * 75)
comparison

**Kesimpulan Evaluasi:**

| Aspek | Decision Tree | Random Forest | Pemenang |
| :--- | :---: | :---: | :---: |
| Akurasi Keseluruhan | ~80% | ~67% | Decision Tree |
| **Recall Stroke** | ~76% | **~84%** | **Random Forest** |

Meskipun akurasi Decision Tree lebih tinggi, **Random Forest memiliki Recall yang lebih baik (84%)** untuk mendeteksi pasien stroke.

Dalam konteks **medis**, Recall jauh lebih penting karena:
- **False Negative** (gagal mendeteksi stroke) → pasien tidak mendapat penanganan → **berbahaya**
- **False Positive** (salah menandai sehat sebagai stroke) → pasien menjalani pemeriksaan tambahan → **aman**

Oleh karena itu, **Random Forest** dipilih sebagai model terbaik untuk deployment.

---
## Tahap 6: Deployment

Model Random Forest yang terpilih disimpan ke dalam file `.pkl` dan digunakan pada aplikasi web **Streamlit** (`app.py`) yang dapat diakses melalui browser.

### 🌐 Link Aplikasi Web Hasil Deployment
Aplikasi web ini telah berhasil di-deploy ke Streamlit Community Cloud dan dapat diakses secara publik oleh dosen/penguji melalui tautan berikut:
👉 **[Aplikasi Web Prediksi Risiko Stroke (UAS AI)](https://prediksi-stroke-w6ndculxw5urdradatsjaq.streamlit.app/)**

**Cara Penggunaan Website:**
1. Klik/buka tautan di atas pada browser Anda.
2. Masukkan data medis pasien (Jenis Kelamin, Usia, Riwayat Hipertensi, Riwayat Penyakit Jantung, Status Pernikahan, Jenis Pekerjaan, Tipe Tempat Tinggal, Kadar Glukosa, BMI, dan Status Merokok).
3. Tekan tombol **Prediksi Risiko Stroke**.
4. Aplikasi web akan memproses data tersebut menggunakan model Random Forest terbaik yang telah dilatih pada notebook ini dan memunculkan tingkat risiko stroke beserta probabilitas persentasenya secara real-time.

In [ ]:
# Simpan model dan daftar fitur untuk deployment
feature_columns = list(X.columns)
with open('feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_columns, f)

with open('best_stroke_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

print("File deployment berhasil disimpan:")
print("  1. best_stroke_model.pkl  (Model Random Forest)")
print("  2. feature_columns.pkl    (Daftar kolom fitur)")
print("\nKedua file ini digunakan oleh aplikasi web Streamlit (app.py).")

---
## Kesimpulan Akhir

Proyek prediksi penyakit stroke menggunakan Machine Learning telah berhasil diselesaikan mengikuti kerangka kerja **CRISP-DM**. Berikut ringkasan dari setiap tahap:

| Tahap CRISP-DM | Hasil |
| :--- | :--- |
| **Business Understanding** | Tujuan: memprediksi risiko stroke untuk deteksi dini |
| **Data Understanding** | Dataset 5.110 pasien, 12 fitur. Data sangat imbalanced (hanya 4.9% kasus stroke). Usia dan kadar glukosa tinggi berkorelasi kuat dengan stroke |
| **Data Preparation** | 201 missing values pada BMI diisi dengan median. 5 kolom kategorikal di-encode menjadi numerik (One-Hot Encoding) |
| **Modeling** | Data dibagi 80:20 (Train:Test). Dua model dilatih: Decision Tree dan Random Forest, keduanya dengan `class_weight='balanced'` |
| **Evaluation** | Random Forest dipilih karena Recall tertinggi (84%) — mampu mendeteksi 84 dari 100 pasien yang benar-benar berisiko stroke |
| **Deployment** | Model disimpan sebagai file `.pkl` dan digunakan dalam aplikasi web interaktif Streamlit |

**Model yang dipilih:** Random Forest Classifier dengan Recall 84% untuk kelas Stroke.

> ⚠️ **Disclaimer:** Hasil prediksi model ini hanya bersifat **edukatif** dan bukan diagnosis medis. Selalu konsultasikan kondisi kesehatan dengan tenaga medis profesional.